In [0]:
%run ../00-common/config-01

In [0]:

bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"



from pyspark.sql import functions as F






races_df = spark.table(bronze_table)




races_selected_df = races_df.select(
    F.col("season"),
    F.col("round"),
    F.col("raceName"),
    F.col("date"),
    F.col("circuitId"),
    F.col("ingestion_timestamp"),
    F.col("source_file")
)




races_renamed_df = (
    races_selected_df
        .withColumnsRenamed({
            "circuitId": "circuit_id",
            "raceName": "race_name",
            "date": "race_date"
        })
)



#display(races_renamed_df)




races_distinct_df = races_renamed_df.dropDuplicates(["season","round"])



#display(races_distinct_df)






races_final_df = (
    races_distinct_df
        .withColumn('race_name', F.initcap(F.col("race_name")))
)



#display(races_final_df)




(
    races_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)



display(spark.table(silver_table))